# Feature-group ablation — which feature families work together?

The experiment log (runs 5–24) adds one feature **family** at a time to a pooled candidate set and lets RFE pick individual columns. That design can't answer *which groups work together*, for two reasons it documents repeatedly:

1. **Greedy per-feature selection hides group structure.** When pbp ranks displace box-score ranks with no hold-out change, RFE is telling us they're *interchangeable* — you can't read group synergy off a feature-level selector.
2. **A single 544-game hold-out can't resolve the differences.** BART wobbles ±0.004; accuracy ±4 pts. Most group combinations differ by *less than that*.

This notebook measures the group structure directly:

- **Partition** the ~9,297-column pool into content families (`data_science_utilities/feature_groups/partition.py`, validated to classify every column).
- **Score the full 2^G subset sweep** with a *fixed, regularised* XGBoost per subset — **no per-subset RFE or tuning**, so the *group* effect is isolated from the *selection* effect.
- Evaluate on **season-blocked rolling-origin CV** (not the single hold-out) so every effect carries a spread across folds.
- **Decompose** into per-group **Shapley main effects** and **pairwise interaction indices**: *positive = super-additive (the pair beats the sum of its parts); negative = sub-additive (adds less than its parts)*.

> **Two caveats baked into the reading** (see the closing cell for detail):
> - **Sub-additive ≠ proven redundant.** A negative interaction can mean correlated/redundant families *or* simply that this model's documented ~0.705 ROC-AUC / ~0.219 Brier **ceiling** caps the pair's joint score even when their information is independent. The index can't separate the two — corroborate redundancy with feature correlations.
> - **The cross-fold SE is a lower bound.** Folds share nested training data and overlapping validation windows, so they're positively correlated; treat the SE as a descriptive spread, not a calibrated significance test.
>
> Given seven feature generations on the plateau, the honest prior is mostly **negative/near-zero** interactions among the box/pbp families and near-zero marginal value past the first one or two — consistent with a data/regime ceiling. The one family expected to carry a positive main effect is `market` (absent here unless the dataset is regenerated with `keep_odds=True`).

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    path = os.path.abspath(start or os.getcwd())
    while path != os.path.dirname(path):
        if os.path.isdir(os.path.join(path, 'data_science_utilities')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError('repo root (with data_science_utilities/) not found from ' + os.getcwd())


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from data_science_utilities.feature_groups.partition import partition_features, TARGET
from data_science_utilities.feature_groups.ablation import (
    cross_validated_group_ablation, feature_subset)
from data_science_utilities.feature_groups.scorers import (
    season_rolling_origin_folds, make_xgb_fold_scorer, make_bart_fold_scorer,
    score_predictions, DEFAULT_XGB_PARAMS)

RANDOM_SEED = 32
np.random.seed(RANDOM_SEED)

# ---- config ----
SELECTION_METRIC = 'brier'          # 'brier' (lower=better) | 'roc_auc' (higher=better)
HIGHER_IS_BETTER = SELECTION_METRIC == 'roc_auc'
RANK_ONLY = True                    # champion pool: rank / rank_change columns only
FIRST_TEST_SEASON = 2019            # rolling-origin test seasons (inclusive range)
LAST_TEST_SEASON = 2025
VALID_YEARS = 2                     # early-stopping window before each test season

PARQUET = os.path.join(REPO_ROOT, 'data', 'predict_games', 'input_data',
                       'schedule_and_weekly.parquet')
PROGRESS_LOG = '/tmp/group_ablation_progress.log'
print('repo root :', REPO_ROOT)
print('metric    :', SELECTION_METRIC, '| higher_is_better =', HIGHER_IS_BETTER)
print('watch live:  tail -f', PROGRESS_LOG)

In [ ]:
df = (pd.read_parquet(PARQUET)
      .sample(frac=1, random_state=RANDOM_SEED)
      .reset_index(drop=True))
assert TARGET in df.columns, TARGET
print('rows x cols :', df.shape)
print('seasons     :', int(df['season'].min()), '-', int(df['season'].max()))
print('hold-out games (>=2024):', df[df['season'] >= 2024]['game_id'].nunique())

In [ ]:
all_groups = partition_features(list(df.columns), rank_only=RANK_ONLY)

# context_rest (rest / dome / division / week) is the always-on BASE every model gets.
# market is empty unless the dataset was regenerated with keep_odds=True.
BASE_GROUP = 'context_rest'
base_features = all_groups.get(BASE_GROUP, [])
toggle_groups = {name: cols for name, cols in all_groups.items()
                 if name != BASE_GROUP and len(cols) > 0}

print('base', BASE_GROUP, ':', len(base_features), 'cols')
for name, cols in toggle_groups.items():
    print('  toggle', name.ljust(24), len(cols), 'cols')
print()
print(len(toggle_groups), 'toggleable groups ->', 2 ** len(toggle_groups),
      'subsets per fold')
if len(toggle_groups) > 8:
    print('WARNING: 2^G is large; consider merging families or fewer folds.')
if 'market' not in toggle_groups:
    print()
    print('NOTE: the market (odds) family is ABSENT (keep_odds=False). This sweep can')
    print('only characterise the already-plateaued box/pbp families; it CANNOT speak to')
    print('whether market signal breaks the ceiling. Regenerate with keep_odds=True to')
    print('include it as a toggleable group.')

## Rolling-origin season CV

Each effect is averaged over **expanding-window season folds** (test = one season; the two seasons before it are the early-stopping validation window; everything earlier is train). Folds are defined purely on `season`, so a game's two perspective rows never split across blocks.

> **SE caveat.** The per-fold spread gives a standard error, but these folds are **not independent** — training sets are strictly nested and validation windows overlap between consecutive folds, so the estimates are positively correlated and `std/√k` *under*states the true SE. Read the SE as a descriptive spread (and inspect `result['per_fold']` for the raw range); do **not** treat `|mean/se| > 2` as a calibrated significance test.

**Cost.** The sweep is `2^G × folds` fixed-XGBoost fits (`G` = toggleable groups). With 7 groups and 7 folds that is ~900 fits — minutes to ~1 hour depending on the machine. Watch `/tmp/group_ablation_progress.log`. To shrink: raise `FIRST_TEST_SEASON`, or drop a family from `toggle_groups`.

In [ ]:
folds = season_rolling_origin_folds(
    df, first_test_season=FIRST_TEST_SEASON, last_test_season=LAST_TEST_SEASON,
    valid_years=VALID_YEARS)
print(len(folds), 'rolling-origin folds')
for f in folds:
    print('  test', f['test_season'],
          '| train', len(f['train']), 'valid', len(f['valid']), 'test', len(f['test']))

In [ ]:
def make_logged_fold_scorer(fold, fold_pos):
    train_df = df.loc[fold['train']]
    valid_df = df.loc[fold['valid']]
    test_df = df.loc[fold['test']]
    base_scorer = make_xgb_fold_scorer(
        train_df, valid_df, test_df, target=TARGET, metric=SELECTION_METRIC)
    state = {'n': 0}

    def scorer(features):
        score = base_scorer(features)
        state['n'] += 1
        with open(PROGRESS_LOG, 'a') as fh:
            print('fold', fold_pos + 1, '/', len(folds),
                  'season', fold['test_season'], 'subset', state['n'],
                  'nfeat', len(features), SELECTION_METRIC, round(score, 4), file=fh)
        return score
    return scorer


open(PROGRESS_LOG, 'w').close()
fold_scorers = [make_logged_fold_scorer(f, i) for i, f in enumerate(folds)]

t0 = time.time()
result = cross_validated_group_ablation(
    toggle_groups, base_features, fold_scorers, metric=SELECTION_METRIC,
    on_progress=lambda rec: print('  fold', rec['fold'] + 1, '/', len(folds),
                                  'done @', round(time.time() - t0), 's'))
print('sweep complete in', round(time.time() - t0), 's',
      '(', len(folds), 'folds x', 2 ** len(toggle_groups), 'subsets )')

In [ ]:
def effect_frame(effect_dict):
    frame = pd.DataFrame(
        {name: {'mean': v['mean'], 'se': v['se']}
         for name, v in effect_dict.items()}).T
    return frame.sort_values('mean', ascending=False)


shapley_df = effect_frame(result['shapley']).copy()
shapley_df['mean/se'] = shapley_df['mean'] / shapley_df['se'].replace(0, np.nan)
print('Shapley main effect per group (utility units; positive = beneficial).')
print('mean/se is a ROUGH effect-to-spread ratio only: the cross-fold SE is a lower')
print('bound (correlated folds), so a large ratio is suggestive, not significant.')
shapley_df

In [ ]:
standalone = effect_frame(result['standalone'])[['mean', 'se']]
standalone.columns = ['standalone', 'standalone_se']
loo = effect_frame(result['leave_one_out'])[['mean', 'se']]
loo.columns = ['leave_one_out', 'loo_se']
redundancy = standalone.join(loo)
redundancy['redundancy (standalone - loo)'] = (
    redundancy['standalone'] - redundancy['leave_one_out'])
print('standalone = value alone over base; leave_one_out = marginal given all others.')
print('Large standalone + near-zero leave_one_out = a redundant substitute.')
redundancy.sort_values('standalone', ascending=False)

In [ ]:
names = result['group_names']
mat = pd.DataFrame(0.0, index=names, columns=names)
for pair, v in result['interactions'].items():
    i, j = sorted(pair)
    mat.loc[i, j] = v['mean']
    mat.loc[j, i] = v['mean']
for k in range(len(names)):
    mat.iloc[k, k] = np.nan
arr = mat.to_numpy()

lim = float(np.nanmax(np.abs(arr))) or 1e-6
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(arr, cmap='RdBu', vmin=-lim, vmax=lim)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
for a in range(len(names)):
    for b in range(len(names)):
        if a != b:
            ax.text(b, a, format(arr[a, b], '+.4f'),
                    ha='center', va='center', fontsize=7)
ax.set_title('Pairwise interaction (' + SELECTION_METRIC +
             ', utility units)\nblue = super-additive, red = sub-additive')
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

inter_tbl = pd.DataFrame([
    {'pair': ' + '.join(sorted(p)), 'interaction': v['mean'], 'se': v['se']}
    for p, v in result['interactions'].items()]).sort_values('interaction')
print('Most sub-additive (negative) at top, most super-additive (positive) at bottom.')
print('Reminder: negative = redundancy OR metric saturation (the ceiling) — not proof')
print('of redundancy on its own; corroborate with feature correlations.')
inter_tbl

In [ ]:
score_rows = []
for subset, mean_score in result['mean_scores'].items():
    score_rows.append({
        'groups': ' + '.join(sorted(subset)) if subset else '(base only)',
        'num_groups': len(subset),
        'mean_' + SELECTION_METRIC: mean_score})
score_df = pd.DataFrame(score_rows).sort_values(
    'mean_' + SELECTION_METRIC, ascending=not HIGHER_IS_BETTER).reset_index(drop=True)

full_key = frozenset(result['group_names'])
print('full set  :', round(result['mean_scores'][full_key], 4))
print('base only :', round(result['mean_scores'][frozenset()], 4))
print()
print('Best subsets:')
display(score_df.head(8))
print('Worst subsets:')
display(score_df.tail(5))

## Confirmation — fixed run-5 hold-out and BART

The rolling-origin sweep above is the structural story. Here we spotlight the standout subsets on the **fixed run-5 split** (train `<2022`, early-stop `2022–2023`, hold-out `2024+2025`).

> **These are NOT directly comparable to the README run table.** This harness uses a *fixed, untuned* regularised XGBoost (no per-subset RFE or hyperparameter search), so the absolute hold-out level is **lower by design** than the README's tuned models. Read the **deltas between subsets**, not the absolute numbers against the README. The optional BART pass (`RUN_BART = True`, slow) trains on `<2024` with no early-stopping slice, so it sees the 2022–2023 seasons that XGBoost holds out — another reason to compare *signs/deltas*, not levels.

In [ ]:
from xgboost import XGBClassifier

train5 = df[df['season'] < 2022]
valid5 = df[(df['season'] >= 2022) & (df['season'] < 2024)]
holdout5 = df[df['season'] >= 2024]
y_holdout5 = holdout5[TARGET].to_numpy()


def xgb_holdout_both(features):
    """Fit once on the run-5 split, return (hold-out ROC-AUC, hold-out Brier)."""
    model = XGBClassifier(**DEFAULT_XGB_PARAMS)
    model.fit(train5[features], train5[TARGET],
              eval_set=[(valid5[features], valid5[TARGET])], verbose=False)
    proba = model.predict_proba(holdout5[features])[:, 1]
    return (score_predictions(y_holdout5, proba, 'roc_auc'),
            score_predictions(y_holdout5, proba, 'brier'))


# Spotlight subsets: base, full, top single family, most super-additive pair.
# These are picked by point estimate only -- with 21 pairs and a lower-bound SE this is
# a DESCRIPTIVE spotlight, not a tested discovery. On the documented plateau, "no pair
# clears the noise" is the EXPECTED outcome; check the pair's se in inter_tbl above
# before reading 'best pair' as real synergy.
shapley_sorted = sorted(result['shapley'].items(),
                        key=lambda kv: kv[1]['mean'], reverse=True)
top_group = shapley_sorted[0][0]
best_pair = max(result['interactions'].items(), key=lambda kv: kv[1]['mean'])[0]
print('spotlight (point-estimate only; confirm against per-fold spread):')
print('  top shapley group :', top_group)
print('  best pair         :', ' + '.join(sorted(best_pair)),
      '| interaction', round(result['interactions'][best_pair]['mean'], 4),
      '+/-', round(result['interactions'][best_pair]['se'], 4))

key_subsets = {
    '(base only)': frozenset(),
    'full set': frozenset(result['group_names']),
    'top shapley: ' + top_group: frozenset([top_group]),
    'best pair: ' + ' + '.join(sorted(best_pair)): frozenset(best_pair),
}

rows = []
for label, subset in key_subsets.items():
    feats = feature_subset(base_features, toggle_groups, subset)
    auroc, brier = xgb_holdout_both(feats)
    rows.append({'subset': label, 'n_features': len(feats),
                 'holdout_roc_auc': round(auroc, 4), 'holdout_brier': round(brier, 4)})
pd.DataFrame(rows)

In [ ]:
RUN_BART = False  # set True to confirm standout subsets on BART (~1-3 min each)

if RUN_BART:
    bart_train = pd.concat([train5, valid5])  # BART needs no early-stopping slice
    bart_holdout = make_bart_fold_scorer(
        bart_train, holdout5, target=TARGET, metric='brier')
    # one cheap warm-up to dodge the PyTensor compile-cache race noted in the README
    _ = bart_holdout(feature_subset(base_features, toggle_groups, frozenset()))
    bart_rows = []
    for label, subset in key_subsets.items():
        feats = feature_subset(base_features, toggle_groups, subset)
        bart_rows.append({'subset': label, 'n_features': len(feats),
                          'bart_holdout_brier': round(bart_holdout(feats), 4)})
    display(pd.DataFrame(bart_rows))
else:
    print('RUN_BART = False — set True to confirm the standout subsets on BART (slow).')

## Reading the result

- **Shapley main effect** (utility units, positive = beneficial): how much each family adds, averaged over all contexts of the others. The `mean/se` column is a rough effect-to-spread ratio — but the SE is a **lower bound** (correlated folds), so a large ratio is suggestive, not significant.
- **standalone vs leave-one-out**: a family with a large standalone value but a near-zero leave-one-out delta adds nothing once the others are present. That is consistent with a **redundant substitute** — but on a saturating metric it is *also* what a ceiling produces, so treat it as "no marginal value here", not proof of informational redundancy.
- **Pairwise interaction**: blue (positive) = **super-additive**; red (negative) = **sub-additive**. Sub-additivity has two indistinguishable causes — genuine redundancy *or* the metric ceiling capping the pair's joint score. Expect mostly red / near-zero among the box/pbp families; to tell redundancy from saturation, check whether the two families' columns are actually correlated (and/or re-run in logit/log-loss space, which saturates less).

### Extending
- **Market signal**: regenerate the dataset with `keep_odds=True` (the `schedule` collector supports it) and re-run — `market` enters as a toggleable group and is the most likely source of a positive main effect / super-additive interaction (genuinely new information vs the box/pbp families).
- **Disambiguate redundancy from saturation**: pair this heatmap with a feature-correlation map between the families, or switch `SELECTION_METRIC` to `log_loss` (less ceiling-bound than ROC-AUC) and check whether the negative interactions shrink.
- **Finer FORM axis**: `partition.form_tags` / `is_rank_only_kept` let you toggle the recency (EWMA/rolling) and aggregated-value axes the same way (set `RANK_ONLY = False` and split by `form_tags`).
- **BART-matched selection**: the estimators disagree about which correlated families they exploit (runs 11/12, 15/16); re-running the sweep with a BART fold scorer would test whether the interaction signs hold for the champion estimator.